# OpenTSLM on Context is Key: free test run in Google Colab

This notebook runs a **small test** of our training pipeline on a free Colab GPU, so you can see every step work
before renting a Nebius machine. Run the cells top to bottom with **Shift + Enter**.

Before you start: **Runtime > Change runtime type > T4 GPU**, then Save.

What each step does is written above it. Nothing here costs money. The real training runs on Nebius later.

## 1. Check that we have a GPU
You should see a table that names a **Tesla T4**. If it says "command not found", go back and pick the T4 GPU runtime.

In [ ]:
!nvidia-smi

## 2. Download the OpenTSLM code and install it
This copies the OpenTSLM project from GitHub into Colab and installs the Python libraries it needs. Takes 2 to 5 minutes.
If Colab asks you to **restart the session** at the end, click Restart and continue with the next cell.

In [ ]:
!git clone -q https://github.com/StanfordBDHG/OpenTSLM.git /content/OpenTSLM
%cd /content/OpenTSLM
!pip install -q -e .

## 3. Install TimeNet with your Context is Key connector (PR 788)
Until PR 788 is merged, we install TimeNet from your pull request.
It also removes `torchao`, a library Colab preinstalls that is too old and breaks the LoRA step.

In [ ]:
%cd /content/OpenTSLM
!git clone -q https://github.com/OpenTSLM/TimeNet.git /content/TimeNet
!cd /content/TimeNet && git fetch -q origin pull/788/head:cik && git checkout -q cik
!pip install -q -e /content/TimeNet/packages/timenet -e /content/TimeNet/packages/timenet-connectors
# Colab ships an old torchao that breaks LoRA ("torchao >0.16 required"). We don't need it.
!pip uninstall -y -q torchao

### Restart the session (required)
Colab only sees the newly installed libraries after a restart. Run the cell below: Colab will say the session crashed
or restarted. That is expected. Then carry on with step 4. Do **not** re-run steps 1 to 3.

In [ ]:
import os
os.kill(os.getpid(), 9)   # restarts the Python session so the new installs load

## 4. Upload our code
Click **Choose files** and pick `opentslm-cik.zip` (the file from the project). The cell unzips it and copies each file
to where OpenTSLM expects it.

Check the last line it prints: `Code version: 2026-11-21 extra-data v5` (or newer). If it shows an older version, or none, you uploaded an old zip.

In [ ]:
%cd /content/OpenTSLM
from google.colab import files
import glob, os, shutil
uploaded = files.upload()   # pick opentslm-cik.zip (Colab may rename it "opentslm-cik (1).zip"; that's fine)
zip_name = next(n for n in uploaded if n.endswith(".zip"))
shutil.rmtree("/content/opentslm-cik", ignore_errors=True)          # throw away any older copy
!unzip -o -q "{zip_name}" -d /content/
for old in glob.glob("opentslm-cik*.zip"):
    os.remove(old)                                                     # so an old zip can't be picked up again
!mkdir -p src/opentslm/time_series_datasets/cik
!touch src/opentslm/time_series_datasets/cik/__init__.py
!cp /content/opentslm-cik/cik_core.py /content/opentslm-cik/CiKQADataset.py /content/opentslm-cik/cik_rationales.jsonl src/opentslm/time_series_datasets/cik/
!cp /content/opentslm-cik/train_cik.py /content/opentslm-cik/eval_cik.py /content/opentslm-cik/make_rationales.py /content/opentslm-cik/VERSION .
print("Code version:", open("VERSION").read().strip())   # should match the version in the project's README


## 5. Log in to Hugging Face
Paste a **read token** from huggingface.co/settings/tokens. Your Llama access is linked to this login,
so the model download in step 7 works.

In [ ]:
%cd /content/OpenTSLM
from huggingface_hub import notebook_login
notebook_login()

## 6. Put Context is Key into TimeNet
This runs your connector: it downloads CiK from Hugging Face, converts it to TimeNet format and saves it
in Colab's local TimeNet registry. The training script then reads it from there.

In [ ]:
%cd /content/OpenTSLM
!timenet-build build servicenow/context-is-key --no-isolation
import timenet
ds = timenet.TimeNet().load("servicenow/context-is-key@1.0.0")
print(len(ds.get_all()), "forecasting tasks in TimeNet")   # expect 355

### 6b. Only if step 6 failed: use the CiK file directly
Skip this if step 6 printed 355. Otherwise, click **Choose files**, pick the CiK parquet file (`test-00000-of-00001.parquet`), and in steps 7 to 9 add `--source hf --data data/cik.parquet` to every `python train_cik.py` and `python eval_cik.py` line.

In [ ]:
%cd /content/OpenTSLM
from google.colab import files
!mkdir -p data
up = files.upload()   # pick test-00000-of-00001.parquet
!mv "{list(up)[0]}" data/cik.parquet
!ls -lh data/cik.parquet

## 7. Smoke test: train on 8 examples for 1 epoch
This only checks that everything runs. Expect a few minutes, most of it downloading the models.
Look for `Loaded 355 CiK instances from TimeNet`, then a train and validation **loss**, then `saved new best model`.

If you see an error that mentions **bfloat16** or **BF16**, the free T4 GPU cannot run this model type. The test
then has to run on Nebius. Copy the error into the project thread.

In [ ]:
%cd /content/OpenTSLM
!python train_cik.py --out results/smoke --max_samples 8 --epochs 1 --batch_size 1

## 8. Look at what the (barely trained) model writes
After 1 epoch on 8 examples the forecasts will be poor. The point is to see the full loop run:
data in, forecast out, score computed.

After only 8 examples, `parsed ok: False` and a flat line of model numbers are normal: the model has not yet learned
to answer in the `Forecast: ...` format, so the script falls back to the average. The `wrote` line shows what it actually
wrote. The real training run is what teaches the format.

In [ ]:
%cd /content/OpenTSLM
!python eval_cik.py --ckpt results/smoke/best_model.pt --out results/smoke --max_samples 3 --num_samples 1
import json
for line in open("results/smoke/test_predictions.jsonl"):
    p = json.loads(line)
    print(p["task_name"], "| parsed ok:", p["parsed_ok"])
    print("  model :", [round(v, 2) for v in list(p["forecast_median"].values())[0][:8]])
    print("  truth :", [round(v, 2) for v in list(p["truth"].values())[0][:8]])
    print("  wrote :", " ".join(p["raw_output"].split())[:300])

## 9. Optional: a short real run (about 30 to 60 minutes on a T4)
Only if the smoke test worked and you want a first real number before Nebius is ready. Colab may disconnect if
the tab sits idle, so keep it open. This trains on the full training split, with the ready-made explanations.

In [ ]:
%cd /content/OpenTSLM
!python train_cik.py --out results/t4 --batch_size 1 --max_history 256
!python eval_cik.py --ckpt results/t4/best_model.pt --out results/t4 --max_history 256 --num_samples 3
!cat results/t4/metrics.json

## 10. Download your results
Colab deletes everything when the session ends, so save the results folder to your Mac.

In [ ]:
%cd /content/OpenTSLM
from google.colab import files
!zip -q -r results.zip results
files.download("results.zip")